# 8차시 실습 — 공정 이상과 원인 후보 찾기

강의에서 배운 순서대로 위에서 아래로 실행합니다(셀을 클릭하고 **Shift + Enter**).
`____`(밑줄 네 개)로 비어 있는 곳만 채우면 됩니다.

- 오늘 데이터는 교육용으로 만든 **가상 데이터**입니다(450건의 공정 기록).
- `합격여부` 열은 **1 = 합격, -1 = 불합격**입니다. 오늘은 `== -1`로 불합격을 자주 고릅니다.
- 위 셀을 건너뛰면 `NameError`가 납니다. 이상하면 "커널 → 다시 시작 후 모두 실행".

## 1단계. 데이터 읽고 불합격 건수 확인하기
CSV를 읽어 `df`에 저장하고, 표의 크기(행, 열)를 확인합니다.

In [ ]:
import pandas as pd

# TODO: CSV 파일을 읽는 함수 이름을 채우세요.
df = pd.____("../../data/weekly/week08/week08_anomaly_root_cause.csv", encoding="utf-8-sig")
print(df.shape)

불합격(-1)인 행만 골라 개수와 비율을 셉니다(4차시 조건 검색).

**주의**: 같다는 비교는 `=`가 아니라 `==`입니다.

In [ ]:
# TODO: 불합격을 뜻하는 값을 채우세요.
fail_all = df[df["합격여부"] == ____]
print("불합격 건수:", len(fail_all))
print(f"전체 불합격률: {len(fail_all) / len(df) * 100:.1f}%")

## 2단계. 그림 먼저, 점수는 나중 — 온도와 두께
먼저 산점도를 그려 점들의 모양을 봅니다. 완성된 코드이니 실행만 하세요.

- 오른쪽 위로 올라가나요(같이 오른다)? 오른쪽 아래로 내려가나요(반대로 움직인다)? 구름처럼 퍼져 있나요(관계없다)?
- **주의**: 한글이 네모(□)로 보이면 맨 위 `plt.rcParams` 두 줄을 실행했는지 확인하세요.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"   # 한글 글꼴(Windows). Mac은 "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False      # 음수 부호(-)가 네모로 깨지지 않게

plt.scatter(df["온도_섭씨"], df["두께_nm"], alpha=0.5)
plt.xlabel("온도_섭씨")
plt.ylabel("두께_nm")
plt.title("온도와 두께")
plt.show()

이제 같은 관계를 점수(상관계수, -1~1)로 확인합니다. `A.corr(B)`는 A와 B가 함께 움직이는 정도를 알려 줍니다.

In [ ]:
# TODO: 상관계수를 계산하는 메서드 이름을 채우세요.
r_temp = df["온도_섭씨"].____(df["두께_nm"])
print("온도-두께 상관계수:", round(r_temp, 2))

## 3단계. 관계없는 변수와 비교하기 — 습도와 두께
같은 두께를 이번에는 습도와 짝지어 봅니다. 습도는 두께와 관계없게 만든 비교용 변수입니다.
2단계 그림과 모양이 어떻게 다른지 비교하세요.

In [ ]:
plt.scatter(df["습도_pct"], df["두께_nm"], alpha=0.5)
plt.xlabel("습도_pct")
plt.ylabel("두께_nm")
plt.title("습도와 두께")
plt.show()

**주의**: 2단계 코드와 모양이 같습니다. 바뀌는 것은 첫 번째 열 이름뿐입니다.

In [ ]:
# TODO: 습도 열 이름을 채우세요.
r_humid = df["____"].corr(df["두께_nm"])
print("습도-두께 상관계수:", round(r_humid, 2))

**생각해 보기**: 온도-두께 점수와 습도-두께 점수를 나란히 놓으면 무엇을 말할 수 있나요?

## 4단계. 정상 범위를 벗어난 기록 찾기 — 진공도
진공도의 학습용 정상 범위는 **4.5~5.5 mTorr**입니다. 범위 밖과 범위 안 기록을 나눠 셉니다.

- 범위 밖: 4.5보다 작거나 **또는** 5.5보다 크다 → `|` (Shift + `\`, 한글 키보드는 `₩` 키)
- 범위 안: 4.5 이상 **그리고** 5.5 이하 → `&`
- **주의**: 조건 하나하나를 괄호 `( )`로 감쌉니다. `or`, `and`를 쓰면 `ValueError`가 납니다.

In [ ]:
# TODO: "또는"을 뜻하는 기호를 채우세요.
vac_out = df[(df["진공도_mTorr"] < 4.5) ____ (df["진공도_mTorr"] > 5.5)]
vac_in = df[(df["진공도_mTorr"] >= 4.5) & (df["진공도_mTorr"] <= 5.5)]

print(f"범위 밖: {len(vac_out)}건, 범위 안: {len(vac_in)}건")

두 집단 각각에서 불합격 비율을 구합니다. (불합격 건수) ÷ (그 집단 건수) × 100 입니다.

In [ ]:
# TODO: 범위 안 집단의 이름을 채우세요.
fail_out = len(vac_out[vac_out["합격여부"] == -1]) / len(vac_out) * 100
fail_in = len(vac_in[vac_in["합격여부"] == -1]) / len(____) * 100

print(f"범위 밖 불합격률: {fail_out:.1f}%")
print(f"범위 안 불합격률: {fail_in:.1f}%")

### 직접 해보기 — 온도도 같은 방법으로
진공도 결과가 정말 "특별한지" 알려면 다른 변수와 비교해야 합니다. 온도의 학습용 정상 범위 **295~305℃**로 같은 계산을 합니다.

- 첫 셀의 빈칸: 정상 범위의 아래·위 숫자
- 둘째 셀의 빈칸: "같다"를 뜻하는 비교 기호(진공도 코드와 같은 모양)

In [ ]:
# TODO: 온도의 정상 범위 숫자 두 개를 채우세요.
temp_out = df[(df["온도_섭씨"] < ____) | (df["온도_섭씨"] > ____)]
temp_in = df[(df["온도_섭씨"] >= 295) & (df["온도_섭씨"] <= 305)]

print(f"범위 밖: {len(temp_out)}건, 범위 안: {len(temp_in)}건")

In [ ]:
# TODO: 불합격(-1)을 고르는 비교 기호를 채우세요.
temp_fail_out = len(temp_out[temp_out["합격여부"] ____ -1]) / len(temp_out) * 100
temp_fail_in = len(temp_in[temp_in["합격여부"] == -1]) / len(temp_in) * 100

print(f"범위 밖 불합격률: {temp_fail_out:.1f}%")
print(f"범위 안 불합격률: {temp_fail_in:.1f}%")

## 5단계. 원인 후보 문장 쓰기
4단계 결과(진공도와 온도)를 비교해 아래 틀로 한 문장을 완성하세요. 숫자를 꼭 넣습니다.

> ____가 정상 범위를 벗어나면 불합격률이 ____%에서 ____%로 높아졌다. 같은 방법으로 본 온도는 ____%와 ____%로 큰 차이가 없었다.
> 따라서 ____는 불합격의 원인 후보로 더 살펴볼 만하다. 다만 원인이라고 단정하려면 추가 확인이 필요하다.

**주의**: "원인이다"가 아니라 "원인 후보"라고 씁니다.

**나의 문장**: (여기에 적어보세요)

### 도전 문제 — 압력도 같은 방법으로 (빠른 학습자용)
압력_Pa의 학습용 정상 범위 **995~1025**로 범위 밖/안 불합격률을 비교합니다. 진공도만큼 차이가 나는지 보세요.

- 첫 셀의 빈칸: 4단계 `vac_out` 줄에서 두 조건을 잇는 기호
- 둘째 셀의 빈칸: 범위 밖 집단의 이름(4단계 `fail_out` 줄과 같은 모양)
- **주의**: 압력_Pa는 교육용 가상 값이라 값의 크기를 실제 공정 압력으로 해석하지 않습니다.

In [ ]:
# TODO: 두 조건을 잇는 기호를 채우세요.
pres_out = df[(df["압력_Pa"] < 995) ____ (df["압력_Pa"] > 1025)]
pres_in = df[(df["압력_Pa"] >= 995) & (df["압력_Pa"] <= 1025)]

print(f"범위 밖: {len(pres_out)}건, 범위 안: {len(pres_in)}건")

In [ ]:
# TODO: 범위 밖 집단의 이름을 채우세요.
pres_fail_out = len(pres_out[pres_out["합격여부"] == -1]) / len(____) * 100
pres_fail_in = len(pres_in[pres_in["합격여부"] == -1]) / len(pres_in) * 100

print(f"범위 밖 불합격률: {pres_fail_out:.1f}%")
print(f"범위 안 불합격률: {pres_fail_in:.1f}%")

## 6단계. AI에게 질문하며 더 알아보기
아래 질문 중 하나 이상을 AI 챗봇에게 물어보고, 새로 알게 된 점을 다음 셀에 한두 줄로 적으세요.

- "상관관계와 인과관계는 어떻게 다른가요? 아이스크림 말고 다른 생활 예시로 설명해주세요."
- "상관계수가 0에 가까운데도 두 값 사이에 관계가 있을 수 있나요? 예를 들어주세요."
- "관계를 판단할 때 왜 관계없는 변수와 비교해 보는 게 도움이 되나요?"

**주의**: AI의 답도 틀릴 수 있습니다. 강의 내용과 비교해 보세요.

**내가 새로 알게 된 점**: (여기에 적어보세요)

## 7단계. AI에게 코드 생성 요청하고 직접 실행해보기
AI에게 아래처럼 요청하고, 받은 코드를 다음 셀에 붙여넣어 실행하세요.

> df라는 DataFrame에서 온도_섭씨, 압력_Pa, 진공도_mTorr, 진동_mm_s, 냉각수온도_섭씨, 습도_pct 열 각각과 두께_nm 열의 상관계수를 for 반복문으로 하나씩 계산해 출력하는 파이썬 코드를 만들어줘.

- 강의 예시 코드를 실행하면 온도만 0.71이고 나머지는 모두 0.1보다 작습니다. 내 결과와 비교해 보세요.

## 8단계. 연습문제 — EQ-02의 진동은 불합격과 관계있을까
먼저 설비별 평균 진동을 구합니다(7차시 `groupby`).

In [ ]:
# TODO: 평균을 구하는 메서드 이름을 채우세요.
print(df.groupby("설비번호")["진동_mm_s"].____().round(3))

진동이 가장 높은 설비만 골라 불합격률을 구하고, 1단계의 전체 불합격률과 비교하세요.

**주의**: 설비 번호를 틀리게 쓰면(예: 하이픈 빠뜨림) 고른 행이 0개가 되어 `ZeroDivisionError: division by zero`가 납니다.

In [ ]:
# TODO: 진동이 가장 높았던 설비 번호를 채우세요.
eq02 = df[df["설비번호"] == "____"]
eq02_fail = len(eq02[eq02["합격여부"] == -1]) / len(eq02) * 100
print(f"EQ-02 불합격률: {eq02_fail:.1f}%")

**생각해 보기**: EQ-02의 진동은 높지만 불합격률은 전체와 비교해 어떤가요? 진동을 불합격의 원인 후보로 올려도 될까요?

## 9단계. AI로 재미있는 미니 프로그램 만들기
AI에게 아래처럼 요청하고, 받은 코드를 다음 셀에 붙여넣어 실행하세요.

> 변수 쌍 이름과 상관계수가 담긴 딕셔너리를 받아서, 상관계수의 절댓값이 0.7 이상이면 '강함', 0.3 이상이면 '중간', 그보다 작으면 '약함'이라고 판정하고, 절댓값 크기만큼 ■ 막대를 그려 출력하는 파이썬 코드를 만들어줘.

- `abs()`(절댓값), `.items()`는 구경만 하세요. 판정 기준은 강의의 "학습용 기준" 표와 같습니다.